In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import linregress

In [2]:
nav = pd.read_csv("../data/raw/02_nav_history.csv")

benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")

fund_master = pd.read_csv("../data/raw/07_scheme_performance.csv")

In [3]:
print(nav.shape)
print(benchmark.shape)
print(fund_master.shape)

(46000, 3)
(8050, 3)
(40, 19)


In [4]:
# Convert date columns

nav["date"] = pd.to_datetime(nav["date"])
benchmark["date"] = pd.to_datetime(benchmark["date"])

In [5]:
nifty50 = benchmark[benchmark["index_name"] == "NIFTY50"].copy()

nifty50.head()

,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [6]:
nifty50 = benchmark[benchmark["index_name"] == "NIFTY50"].copy()

nifty50.head()

,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [7]:
nav = nav.sort_values(["amfi_code", "date"])

nav.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [8]:
nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"].pct_change()
)

nav.head(10)

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210
5755,100016,2022-01-10,510.7136,-0.008639
5756,100016,2022-01-11,513.5542,0.005562
5757,100016,2022-01-12,512.3195,-0.002404
5758,100016,2022-01-13,510.2445,-0.004050
5759,100016,2022-01-14,514.3636,0.008073


In [9]:
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()

In [10]:
nav.head(10)

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210
5755,100016,2022-01-10,510.7136,-0.008639
5756,100016,2022-01-11,513.5542,0.005562
5757,100016,2022-01-12,512.3195,-0.002404
5758,100016,2022-01-13,510.2445,-0.004050
5759,100016,2022-01-14,514.3636,0.008073


In [11]:
nav["year"] = nav["date"].dt.year

nav.head()

,amfi_code,date,nav,daily_return,year
5750,100016,2022-01-03,520.4608,NaN,2022
5751,100016,2022-01-04,515.0971,-0.010306,2022
5752,100016,2022-01-05,521.7239,0.012865,2022
5753,100016,2022-01-06,515.7880,-0.011377,2022
5754,100016,2022-01-07,515.1639,-0.001210,2022


In [12]:
cagr_list = []

for code in nav["amfi_code"].unique():

    fund = nav[nav["amfi_code"] == code].sort_values("date")

    start_nav = fund["nav"].iloc[0]
    end_nav = fund["nav"].iloc[-1]

    years = (fund["date"].iloc[-1] - fund["date"].iloc[0]).days / 365.25

    cagr = ((end_nav / start_nav) ** (1 / years) - 1) * 100

    cagr_list.append([code, cagr])

cagr_df = pd.DataFrame(cagr_list, columns=["amfi_code", "cagr_pct"])

cagr_df.head()

,amfi_code,cagr_pct
0,100016,2.637074
1,100025,4.458210
2,100033,30.123153
3,101206,23.538361
4,101207,7.938765


In [13]:
sharpe_df = nav.groupby("amfi_code")["daily_return"].agg(["mean", "std"]).reset_index()

risk_free_rate = 0.065

sharpe_df["sharpe_ratio"] = (
    (sharpe_df["mean"] * 252 - risk_free_rate) /
    (sharpe_df["std"] * np.sqrt(252))
)

sharpe_df.head()

,amfi_code,mean,std,sharpe_ratio
0,100016,0.000142,0.009164,-0.201517
1,100025,0.000170,0.002460,-0.567095
2,100033,0.001080,0.011929,1.093699
3,101206,0.000852,0.009177,1.027213
4,101207,0.000424,0.016251,0.162661


In [14]:
top_sharpe = sharpe_df.sort_values("sharpe_ratio", ascending=False)

top_sharpe.head(10)

,amfi_code,mean,std,sharpe_ratio
34,148567,0.001074,0.008941,1.448291
30,120843,0.001082,0.010008,1.306744
36,148569,0.001124,0.011134,1.234930
19,119551,0.000917,0.008656,1.208267
25,120505,0.001161,0.012152,1.180101
38,149323,0.001055,0.011179,1.132122
2,100033,0.001080,0.011929,1.093699
9,118632,0.000865,0.008913,1.081659
3,101206,0.000852,0.009177,1.027213
24,120504,0.000843,0.009048,1.026524


In [15]:
fig = px.bar(
    top_sharpe.head(10),
    x="amfi_code",
    y="sharpe_ratio",
    title="Top 10 Funds by Sharpe Ratio",
    text="sharpe_ratio"
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")

fig.show()

In [16]:
top_sharpe.head(10)

,amfi_code,mean,std,sharpe_ratio
34,148567,0.001074,0.008941,1.448291
30,120843,0.001082,0.010008,1.306744
36,148569,0.001124,0.011134,1.234930
19,119551,0.000917,0.008656,1.208267
25,120505,0.001161,0.012152,1.180101
38,149323,0.001055,0.011179,1.132122
2,100033,0.001080,0.011929,1.093699
9,118632,0.000865,0.008913,1.081659
3,101206,0.000852,0.009177,1.027213
24,120504,0.000843,0.009048,1.026524


In [17]:
sharpe_df.describe()

,amfi_code,mean,std,sharpe_ratio
count,40.000000,40.000000,40.000000,40.000000
mean,120247.000000,0.000631,0.009414,0.537220
std,14534.998667,0.000348,0.004205,0.573689
min,100016.000000,0.000110,0.000311,-0.815567
25%,118632.750000,0.000273,0.008724,0.064696
50%,119551.500000,0.000648,0.009171,0.647043
75%,120842.250000,0.000878,0.011458,1.005304
max,149324.000000,0.001201,0.016251,1.448291


In [18]:
sharpe_df.isna().sum()

amfi_code       0
mean            0
std             0
sharpe_ratio    0
dtype: int64

In [19]:
sharpe_df = sharpe_df.replace([np.inf, -np.inf], np.nan)

sharpe_df = sharpe_df.dropna(subset=["sharpe_ratio"])

top_sharpe = sharpe_df.sort_values("sharpe_ratio", ascending=False)

In [25]:
top10 = top_sharpe.head(10).copy()

top10["amfi_code"] = top10["amfi_code"].astype(str)

fig = px.bar(
    top10,
    x="amfi_code",
    y="sharpe_ratio",
    text="sharpe_ratio",
    title="Top 10 Funds by Sharpe Ratio"
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")

fig.update_layout(
    xaxis_title="AMFI Code",
    yaxis_title="Sharpe Ratio"
)

fig.show()

In [21]:
top_sharpe.head(10)

,amfi_code,mean,std,sharpe_ratio
34,148567,0.001074,0.008941,1.448291
30,120843,0.001082,0.010008,1.306744
36,148569,0.001124,0.011134,1.234930
19,119551,0.000917,0.008656,1.208267
25,120505,0.001161,0.012152,1.180101
38,149323,0.001055,0.011179,1.132122
2,100033,0.001080,0.011929,1.093699
9,118632,0.000865,0.008913,1.081659
3,101206,0.000852,0.009177,1.027213
24,120504,0.000843,0.009048,1.026524


In [22]:
sharpe_df.describe()

,amfi_code,mean,std,sharpe_ratio
count,40.000000,40.000000,40.000000,40.000000
mean,120247.000000,0.000631,0.009414,0.537220
std,14534.998667,0.000348,0.004205,0.573689
min,100016.000000,0.000110,0.000311,-0.815567
25%,118632.750000,0.000273,0.008724,0.064696
50%,119551.500000,0.000648,0.009171,0.647043
75%,120842.250000,0.000878,0.011458,1.005304
max,149324.000000,0.001201,0.016251,1.448291


In [23]:
sharpe_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   amfi_code     40 non-null     int64  
 1   mean          40 non-null     float64
 2   std           40 non-null     float64
 3   sharpe_ratio  40 non-null     float64
dtypes: float64(3), int64(1)
memory usage: 1.4 KB


In [24]:
print(len(top_sharpe))
print(top_sharpe["sharpe_ratio"].isna().sum())
print(top_sharpe["sharpe_ratio"].max())
print(top_sharpe["sharpe_ratio"].min())

40
0
1.4482911703262604
-0.8155673679088258


In [26]:
downside = nav.copy()

downside["downside_return"] = downside["daily_return"]

downside.loc[
    downside["downside_return"] > 0,
    "downside_return"
] = 0

In [27]:
sortino_df = downside.groupby("amfi_code").agg(
    mean_return=("daily_return", "mean"),
    downside_std=("downside_return", "std")
).reset_index()

risk_free_rate = 0.065

sortino_df["sortino_ratio"] = (
    (sortino_df["mean_return"] * 252 - risk_free_rate)
    /
    (sortino_df["downside_std"] * np.sqrt(252))
)

sortino_df.head()

,amfi_code,mean_return,downside_std,sortino_ratio
0,100016,0.000142,0.005163,-0.357702
1,100025,0.000170,0.001382,-1.009643
2,100033,0.001080,0.006632,1.967316
3,101206,0.000852,0.004989,1.889576
4,101207,0.000424,0.009256,0.285576


In [28]:
top_sortino = sortino_df.sort_values(
    "sortino_ratio",
    ascending=False
)

top_sortino.head(10)

,amfi_code,mean_return,downside_std,sortino_ratio
34,148567,0.001074,0.004962,2.609850
30,120843,0.001082,0.005308,2.463887
36,148569,0.001124,0.006035,2.278106
19,119551,0.000917,0.004662,2.243325
25,120505,0.001161,0.006644,2.158384
38,149323,0.001055,0.006291,2.011887
2,100033,0.001080,0.006632,1.967316
9,118632,0.000865,0.004914,1.961986
3,101206,0.000852,0.004989,1.889576
24,120504,0.000843,0.004947,1.877411


In [29]:
top10 = top_sortino.head(10).copy()

top10["amfi_code"] = top10["amfi_code"].astype(str)

fig = px.bar(
    top10,
    x="amfi_code",
    y="sortino_ratio",
    text="sortino_ratio",
    title="Top 10 Funds by Sortino Ratio"
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside"
)

fig.show()

In [30]:
nifty50 = benchmark[benchmark["index_name"] == "NIFTY50"].copy()

nifty50["date"] = pd.to_datetime(nifty50["date"])

nifty50 = nifty50.sort_values("date")

nifty50["benchmark_return"] = nifty50["close_value"].pct_change()

nifty50.head()

,date,index_name,close_value,benchmark_return
0,2022-01-03,NIFTY50,17492.79,NaN
1,2022-01-04,NIFTY50,17689.64,0.011253
2,2022-01-05,NIFTY50,17835.05,0.008220
3,2022-01-06,NIFTY50,17878.51,0.002437
4,2022-01-07,NIFTY50,17759.15,-0.006676


In [31]:
merged = pd.merge(
    nav,
    nifty50[["date", "benchmark_return"]],
    on="date",
    how="inner"
)

merged.head()

,amfi_code,date,nav,daily_return,year,benchmark_return
0,100016,2022-01-03,520.4608,NaN,2022,NaN
1,100016,2022-01-04,515.0971,-0.010306,2022,0.011253
2,100016,2022-01-05,521.7239,0.012865,2022,0.008220
3,100016,2022-01-06,515.7880,-0.011377,2022,0.002437
4,100016,2022-01-07,515.1639,-0.001210,2022,-0.006676


In [32]:
alpha_beta = []

for code, group in merged.groupby("amfi_code"):

    group = group.dropna()

    if len(group) < 30:
        continue

    slope, intercept, r, p, stderr = linregress(
        group["benchmark_return"],
        group["daily_return"]
    )

    alpha = intercept * 252
    beta = slope

    alpha_beta.append([code, alpha, beta])

alpha_beta_df = pd.DataFrame(
    alpha_beta,
    columns=["amfi_code", "alpha", "beta"]
)

alpha_beta_df.head()

,amfi_code,alpha,beta
0,100016,0.036221,-0.025909
1,100025,0.043189,-0.016176
2,100033,0.272343,-0.011200
3,101206,0.213945,0.033814
4,101207,0.108205,-0.059856


In [33]:
top_alpha = alpha_beta_df.sort_values(
    "alpha",
    ascending=False
)

top_alpha.head(10)

,amfi_code,alpha,beta
21,119598,0.301114,0.074266
39,149324,0.298179,0.132608
25,120505,0.293014,-0.017391
36,148569,0.283473,-0.010201
30,120843,0.272784,-0.008737
2,100033,0.272343,-0.011200
34,148567,0.271150,-0.028133
38,149323,0.265836,0.003479
16,119094,0.259971,-0.059868
19,119551,0.232196,-0.056045


In [34]:
top10 = top_alpha.head(10).copy()

top10["amfi_code"] = top10["amfi_code"].astype(str)

fig = px.bar(
    top10,
    x="amfi_code",
    y="alpha",
    text="alpha",
    title="Top 10 Funds by Alpha"
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="AMFI Code",
    yaxis_title="Alpha"
)

fig.show()

In [35]:
top_beta = alpha_beta_df.sort_values(
    "beta",
    ascending=False
)

top_beta.head(10)

,amfi_code,alpha,beta
39,149324,0.298179,0.132608
21,119598,0.301114,0.074266
26,120506,0.162837,0.047708
28,120841,0.130683,0.041676
8,102887,0.161793,0.040116
9,118632,0.217281,0.036434
3,101206,0.213945,0.033814
24,120504,0.212094,0.017025
37,149322,0.131299,0.014849
17,119095,0.045680,0.013292


In [36]:
top10 = top_beta.head(10).copy()

top10["amfi_code"] = top10["amfi_code"].astype(str)

fig = px.bar(
    top10,
    x="amfi_code",
    y="beta",
    text="beta",
    title="Top 10 Funds by Beta"
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="AMFI Code",
    yaxis_title="Beta"
)

fig.show()

In [37]:
nav = nav.sort_values(["amfi_code", "date"])

nav["running_max"] = nav.groupby("amfi_code")["nav"].cummax()

nav["drawdown"] = (nav["nav"] / nav["running_max"]) - 1

nav.head()

,amfi_code,date,nav,daily_return,year,running_max,drawdown
5750,100016,2022-01-03,520.4608,NaN,2022,520.4608,0.000000
5751,100016,2022-01-04,515.0971,-0.010306,2022,520.4608,-0.010306
5752,100016,2022-01-05,521.7239,0.012865,2022,521.7239,0.000000
5753,100016,2022-01-06,515.7880,-0.011377,2022,521.7239,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210,2022,521.7239,-0.012574


In [38]:
drawdown_df = nav.groupby("amfi_code").agg(
    max_drawdown=("drawdown", "min")
).reset_index()

drawdown_df.head()

,amfi_code,max_drawdown
0,100016,-0.247344
1,100025,-0.043083
2,100033,-0.162172
3,101206,-0.112916
4,101207,-0.354469


In [39]:
worst_drawdown = drawdown_df.sort_values(
    "max_drawdown"
)

worst_drawdown.head(10)

,amfi_code,max_drawdown
22,119599,-0.525742
17,119095,-0.516778
4,101207,-0.354469
39,149324,-0.311719
21,119598,-0.287060
7,102886,-0.280011
0,100016,-0.247344
29,120842,-0.240035
11,118634,-0.233449
15,119093,-0.217514


In [40]:
top10 = worst_drawdown.head(10).copy()

top10["amfi_code"] = top10["amfi_code"].astype(str)

fig = px.bar(
    top10,
    x="amfi_code",
    y="max_drawdown",
    text="max_drawdown",
    title="Top 10 Worst Maximum Drawdowns"
)

fig.update_traces(
    texttemplate="%{text:.2%}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="AMFI Code",
    yaxis_title="Maximum Drawdown"
)

fig.show()

In [41]:
nav["date"] = pd.to_datetime(nav["date"])
nifty50["date"] = pd.to_datetime(nifty50["date"])

In [42]:
nifty50 = nifty50.sort_values("date")
nifty50["benchmark_return"] = nifty50["close_value"].pct_change()

nifty50.head()

,date,index_name,close_value,benchmark_return
0,2022-01-03,NIFTY50,17492.79,NaN
1,2022-01-04,NIFTY50,17689.64,0.011253
2,2022-01-05,NIFTY50,17835.05,0.008220
3,2022-01-06,NIFTY50,17878.51,0.002437
4,2022-01-07,NIFTY50,17759.15,-0.006676


In [43]:
merged = pd.merge(
    nav,
    nifty50[["date", "benchmark_return"]],
    on="date",
    how="inner"
)

merged.head()

,amfi_code,date,nav,daily_return,year,running_max,drawdown,benchmark_return
0,100016,2022-01-03,520.4608,NaN,2022,520.4608,0.000000,NaN
1,100016,2022-01-04,515.0971,-0.010306,2022,520.4608,-0.010306,0.011253
2,100016,2022-01-05,521.7239,0.012865,2022,521.7239,0.000000,0.008220
3,100016,2022-01-06,515.7880,-0.011377,2022,521.7239,-0.011377,0.002437
4,100016,2022-01-07,515.1639,-0.001210,2022,521.7239,-0.012574,-0.006676


In [44]:
correlation_list = []

for code, group in merged.groupby("amfi_code"):
    corr = group["daily_return"].corr(group["benchmark_return"])
    correlation_list.append([code, corr])

correlation_df = pd.DataFrame(
    correlation_list,
    columns=["amfi_code", "correlation"]
)

correlation_df.head()

,amfi_code,correlation
0,100016,-0.022916
1,100025,-0.053297
2,100033,-0.007610
3,101206,0.029866
4,101207,-0.029855


In [45]:
top_corr = correlation_df.sort_values(
    "correlation",
    ascending=False
)

top_corr.head(10)

,amfi_code,correlation
39,149324,0.068691
26,120506,0.040305
28,120841,0.039829
18,119120,0.038892
21,119598,0.038011
5,101208,0.033724
9,118632,0.033135
8,102887,0.032756
3,101206,0.029866
24,120504,0.015252


In [46]:
top10 = top_corr.head(10).copy()

top10["amfi_code"] = top10["amfi_code"].astype(str)

fig = px.bar(
    top10,
    x="amfi_code",
    y="correlation",
    text="correlation",
    title="Top 10 Funds by Correlation with NIFTY50"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="AMFI Code",
    yaxis_title="Correlation"
)

fig.show()